# Chapter 28: Probability Calibration

## Overview
This chapter explores **probability calibration**, ensuring that predicted probabilities match true frequency of outcomes. A calibrated classifier outputs meaningful probabilities, not just well-separated scores.

## Key Concepts
- **Calibration**: Do predicted probabilities match reality?
- **Overconfident**: Model assigns high probability, but outcomes less frequent
- **Underconfident**: Model assigns low probability, but outcomes more frequent
- **Calibration Methods**: Techniques to fix uncalibrated models

## Evolution of Examples

### From Problem to Solution
1. **01_uncalibrated_svm.py**: Train SVM, show miscalibration
2. **02_calibrated_svm.py**: Apply calibration method to fix it
3. **03_compare_calibration.py**: Compare uncalibrated vs calibrated

## What is Calibration?

### Definition
**Model is calibrated if**:
```
When model predicts probability p, outcome occurs with frequency p
```

### Example
Among all predictions where model says p=0.8:
- Calibrated: ~80% of outcomes are positive
- Overconfident: <80% positive (model too sure)
- Underconfident: >80% positive (model not sure enough)

## Calibration Curve (Reliability Diagram)

### Construction
1. Bin predictions by confidence level (0-0.1, 0.1-0.2, ..., 0.9-1.0)
2. For each bin: Calculate mean predicted probability
3. For each bin: Calculate empirical frequency of positive outcomes
4. Plot empirical frequency vs mean predicted probability

### Interpretation
- **Perfect diagonal**: Calibrated (predicted p = observed frequency)
- **Above diagonal**: Overconfident (predicts high p, but low frequency)
- **Below diagonal**: Underconfident (predicts low p, but high frequency)

### Why it Matters
For many applications, actual probabilities matter:
- Medical diagnosis: "80% confidence" should mean 80% will have disease
- Risk assessment: "high risk" should have high actual risk
- Decision-making: Need calibrated probabilities to set thresholds

## Why Models Get Uncalibrated

### Neural Networks
Deep networks often overconfident:
- Large capacity: Can memorize training data
- No explicit probability constraint
- Softmax/sigmoid can output extreme values

### SVM
SVM is not probabilistic:
- Decision boundary is what matters
- Distance to boundary ≠ probability
- Large margin makes probabilities extreme

### Other Models
- Gradient boosting: Can overfit, become overconfident
- Logistic regression: Usually well-calibrated (probabilistic objective)
- Naive Bayes: Can be overconfident with independence assumption

## Calibration Methods

### Method 1: Platt Scaling
Fit logistic regression to model predictions:
```
p_calibrated = sigmoid(a × p_raw + b)
```

**Advantages**:
- Simple, one parameter per class
- Fast, minimal computation
- Effective for SVM

**Disadvantages**:
- Only two parameters (limited flexibility)
- Linear warping in logit space

### Method 2: Isotonic Regression
Fit monotonic function to model predictions:
```
p_calibrated = f(p_raw)   where f is monotonic increasing
```

**Advantages**:
- More flexible than Platt
- Preserves ranking (p_raw order unchanged)
- Non-parametric

**Disadvantages**:
- Requires more data (many parameters)
- Can overfit with small calibration set
- More complex implementation

### Method 3: Temperature Scaling (Neural Networks)
Divide logits by temperature before softmax:
```
p_calibrated = softmax(logits / T)
```

**Advantages**:
- Single parameter (T)
- Preserves ranking
- Great for deep networks

**Disadvantages**:
- Only for networks with logit access
- May not handle multi-class well

## Scikit-learn Calibration

### Usage
```python
from sklearn.calibration import CalibratedClassifierCV

# Original uncalibrated model
clf = SVC(kernel='rbf')

# Wrap with calibration
calibrated_clf = CalibratedClassifierCV(clf, method='sigmoid')
# or method='isotonic'

calibrated_clf.fit(X_train, y_train)
proba_calibrated = calibrated_clf.predict_proba(X_test)
```

### Cross-Validation Strategy
To avoid overfitting:
1. Split: Train set → fit original model
2. Validation set → fit calibration function
3. Test set → evaluate

CalibratedClassifierCV handles this automatically with cross-validation.

## When to Calibrate

### Always Calibrate
- Decisions depend on probability values (not just ranking)
- Medical/financial decisions
- Customer communication ("80% likely")

### Optional
- Only ranking matters (e.g., ranking by risk)
- Fixed threshold decision (above/below)
- AUC or ranking metrics are main concern

### Don't Need
- Already probabilistic model (logistic regression, Naive Bayes)
- Hard predictions (not probabilities)

## Evaluation Metrics for Calibration

### Expected Calibration Error (ECE)
```
ECE = Σᵢ (size_i / n) × |accuracy_i - confidence_i|
```
Average gap between predicted confidence and empirical accuracy.

### Brier Score
```
BS = (1/n) × Σᵢ (yᵢ - pᵢ)²
```
Directly measures probability quality.

### Log Loss
```
LL = -(1/n) × Σᵢ [yᵢ log(pᵢ) + (1-yᵢ) log(1-pᵢ)]
```
Information-theoretic measure.

## Practical Example: SVM

### Problem
SVM trained on data:
- Training accuracy: 95%
- But predictions: 0% or 100% (no intermediate probabilities)
- Not useful for probability-dependent decisions

### Solution
1. Get predictions from SVM.decision_function(X)
2. Split data: train SVM on 70%, calibrate on 30%
3. Apply Platt scaling: p = sigmoid(a × decision + b)
4. Now outputs probabilities between 0 and 1

### Verification
- Plot calibration curve: should be closer to diagonal
- Brier score should improve
- Ranking preserved (AUC unchanged)

## Key Takeaways
1. Calibration: predicted p should match observed frequency
2. Many models (SVM, neural nets) are overconfident
3. Calibration curve (reliability diagram) visualizes problem
4. Platt scaling: simple, effective for SVM
5. Isotonic regression: more flexible, requires more data
6. Temperature scaling: for neural networks
7. Calibrate when probability values matter, not just ranking
8. Use cross-validation to avoid overfitting calibration function